# Latent Mesh — Multi-Notebook Training

Train the core on ONE master notebook, while MANY worker notebooks train expert shards in parallel.

**Roles:**
- **Master** (1 only) — trains canvas + latent space, pushes checkpoint to Hub
- **Worker** (many) — pulls master checkpoint, trains 1 expert, pushes back
- **Merge** (run once) — assembles final model from master + all workers

**Before running:** Get a HF token at `huggingface.co/settings/tokens` (Write scope).

In [ ]:
# @title 1. Install dependencies
import os, sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.5', 'transformers', 'datasets', 'numpy',
    'accelerate', 'huggingface-hub', 'tensorboard', 'bitsandbytes'], check=True)
print('Deps installed')

In [ ]:
# @title 2. Clone/update repo
import os, sys, subprocess
WORK = '/content'
os.chdir(WORK)

REPO = 'https://github.com/mohamed-raad/latent-mesh-diffusion'
if not os.path.isdir('latent-mesh-diffusion'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
else:
    os.chdir('latent-mesh-diffusion')
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', 'master'], check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/master'], check=True)
os.chdir(WORK + '/latent-mesh-diffusion')
sys.path.insert(0, 'NoProp/src')
print('Repo ready')

In [ ]:
# @title 3. Login to HuggingFace Hub
from huggingface_hub import login, HfApi
import getpass

HF_TOKEN = getpass.getpass('Enter your HF token:')
HF_REPO_ID = 'mohamed99raad/Latent-Mesh-Model'  # @param {type:"string"}

login(token=HF_TOKEN, add_to_git_credential=True)
os.environ['HF_TOKEN'] = HF_TOKEN

api = HfApi()
api.create_repo(HF_REPO_ID, repo_type='model', private=True, exist_ok=True)
print(f'Logged in as {api.whoami()["name"]}')
print(f'Model repo: {HF_REPO_ID}')

In [ ]:
# @title 4. Select role and run
ROLE = 'Master'  # @param ['Master', 'Worker', 'Merge']
HUB_REPO = HF_REPO_ID  # @param {type:"string"}
print(f'Role: {ROLE}')
print(f'Hub:  {HUB_REPO}')

if ROLE == 'Master':
    !python scripts/run_master.py --hub_repo {HUB_REPO}
elif ROLE == 'Worker':
    !python scripts/run_worker.py --hub_repo {HUB_REPO}
elif ROLE == 'Merge':
    !python scripts/merge_checkpoints.py --hub_repo {HUB_REPO} --out /content/checkpoints/final
    print('\n✓ Merge complete — model ready at /content/checkpoints/final')